# KG1 V81 — Canonicalized submission pipeline

Based on the reverse-engineered Kaggle metric (`docs/KAGGLE_METRIC_ANALYSIS.md`).

Adds to the V80 MEGA pipeline:

1. Dataset canonicalization via `scripts/kg1_sft_format_validator.py`.
2. Prompt engineering that enforces a single terminal `\boxed{}`.
3. Post-processing hook `scripts/kg1_canonicalize_output.py::canonicalize_answer`.
4. Pre-score validation via `scripts/kg1_prescore_rf.prescore_submission` before submit.

Author: FELIPEACASTRO  
Date: 2026-04-22

## Cell 0 — Colab secrets + workspace setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import userdata  # type: ignore
    os.environ['HF_TOKEN'] = userdata.get('HF_KEY')
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception:
    pass

WORKSPACE = Path('/content/kg1-v81')
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)
print('workspace:', WORKSPACE)

## Cell 1 — Clone / pull KG1 repo

In [ ]:
REPO_DIR = WORKSPACE / 'KG1'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', 'claude/competent-shamir',
                    'https://github.com/FELIPEACASTRO/KG1-NVIDIA.git', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('repo:', REPO_DIR)

## Cell 2 — Install deps (V80 MEGA wheel set)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch==2.5.1', 'transformers==4.46.0', 'peft==0.13.2',
                'accelerate==1.0.1', 'datasets==3.0.2', 'bitsandbytes==0.44.1',
                'trl==0.11.4', 'scikit-learn==1.5.2',
                'kaggle==1.6.17', 'huggingface_hub==0.26.2'], check=True)

## Cell 3 — Validate the SFT dataset format

In [ ]:
SFT_PATH = REPO_DIR / 'data' / 'sft' / 'v80_mega.jsonl'
if not SFT_PATH.exists():
    print('WARNING: expected SFT file not found at', SFT_PATH)
else:
    result = subprocess.run([sys.executable, 'scripts/kg1_sft_format_validator.py',
                             '--input', str(SFT_PATH),
                             '--report-json', 'runs/v81_sft_format_report.json',
                             '--report-csv', 'runs/v81_sft_format_errors.csv'],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)

## Cell 4 — Canonicalize labels before training

Rewrites each completion with `canonicalize_answer` using the detected family.
The output JSONL has exactly one terminal `\boxed{}` per row and (for the
equation_transform family) also a `Final answer is:` line to bypass the
nested-brace bug.

In [ ]:
import json
from scripts.kg1_canonicalize_output import canonicalize_answer, detect_family

RAW = SFT_PATH
CLEAN = SFT_PATH.with_name('v81_canonical.jsonl')
count = 0
with RAW.open('r', encoding='utf-8') as fin, CLEAN.open('w', encoding='utf-8') as fout:
    for line in fin:
        row = json.loads(line)
        prompt = row.get('prompt') or row.get('instruction') or ''
        fam = detect_family(prompt)
        completion = row.get('completion') or row.get('answer') or ''
        row['completion'] = canonicalize_answer(completion, family_hint=fam)
        fout.write(json.dumps(row, ensure_ascii=False) + '\n')
        count += 1
print(f'Rewrote {count} rows -> {CLEAN}')

## Cell 5 — Train LoRA adapter (V80 MEGA config, but on the canonical dataset)

Copy your V80 MEGA cell here. The only change: point `dataset_path` to
`v81_canonical.jsonl` and include the format-guard prompt below.

In [ ]:
FORMAT_GUARD = (
    'After you finish reasoning, write exactly one line:\n'
    '  Final answer is: <answer>\n'
    'Then on the next line write only: \\boxed{<answer>}\n'
    'Rules for <answer>:\n'
    '- No \\frac, \\text, \\mathrm, or \\sqrt.\n'
    '- No units (no m, kg, s, etc.).\n'
    '- No thousand separators (no commas).\n'
    '- No scientific notation (no 1e3).\n'
    '- For bit_manipulation: exactly 8 binary digits (zero-padded).\n'
    '- For cipher: lowercase ASCII only, keep spaces.\n'
    '- For roman numerals: uppercase letters only.\n'
)
print(FORMAT_GUARD)

## Cell 6 — Pre-score the trained adapter

Runs a stratified 100-row subset of the train data through the adapter,
applies `canonicalize_answer`, and feeds per-family pass-rates to a seeded
Random-Forest that predicts the Kaggle leaderboard score.

In [ ]:
from scripts.kg1_prescore_rf import prescore_submission

ADAPTER_DIR = REPO_DIR / 'runs' / 'v81_adapter'
report = prescore_submission(str(ADAPTER_DIR), val_subset_size=100)
import json as _json
print(_json.dumps({k: v for k, v in report.items() if k != 'details'}, indent=2))
if report['predicted_kaggle_score'] < 0.80:
    raise RuntimeError(f"Pre-score below 0.80: {report['predicted_kaggle_score']:.3f}")

## Cell 7 — Build submission with post-processing hook

Runs vLLM inference (temperature=0.0, max_tokens=7680) and, before writing the
CSV, wraps every raw output through `canonicalize_answer`.

In [ ]:
import pandas as pd
from scripts.kg1_canonicalize_output import canonicalize_answer, detect_family

# raw_df must come from your vLLM pipeline — one column ``id``, one column
# ``raw_output`` and (optionally) one column ``prompt`` for family auto-detection.
raw_df = pd.read_csv(REPO_DIR / 'runs' / 'v81_raw_predictions.csv')
raw_df['family'] = raw_df['prompt'].apply(detect_family)
raw_df['prediction'] = [
    canonicalize_answer(raw, family_hint=fam)
    for raw, fam in zip(raw_df['raw_output'], raw_df['family'])
]
raw_df[['id', 'prediction']].to_csv('submission.csv', index=False)
print('Wrote submission.csv with canonicalized predictions:', len(raw_df))

## Cell 8 — Submit to Kaggle (with the 99% rule check)

The `feedback_99percent_rule.md` policy says: never submit unless pre-score
predicts an IMPROVEMENT vs the last Kaggle LB. Edit `LAST_KAGGLE_LB` below
before running.

In [ ]:
LAST_KAGGLE_LB = 0.85  # V80 MEGA dgxchen v7
MIN_DELTA = 0.005
if report['predicted_kaggle_score'] < LAST_KAGGLE_LB + MIN_DELTA:
    raise RuntimeError(
        f"Predicted {report['predicted_kaggle_score']:.3f} < last LB {LAST_KAGGLE_LB} + {MIN_DELTA}. Aborting.")
print('Predicted improvement confirmed — safe to submit.')
# subprocess.run(['kaggle', 'competitions', 'submit', '-c',
#                 'nvidia-nemotron-model-reasoning-challenge',
#                 '-f', 'submission.zip', '-m', 'V81 canonicalized'], check=True)